In [13]:
import pandas as pd
from pathlib import Path

import plotly.graph_objects as go

def process_sauna_data(file_path, session_minutes=90, remove_sensor='Sensor2'):
    """
    Clean sauna temperature test data and export cleaned CSV.

    Output filename:
        original_filename_clean.csv
    """

    file_path = Path(file_path)

    # Load CSV
    df = pd.read_csv(file_path)

    # Remove unwanted sensor
    if remove_sensor is not None:
        df = df[df['sensor_label'] != remove_sensor].copy()

    # Calculate control-panel timer
    df['Timer_in_Min'] = session_minutes - df['elapsed_time_in_Min']

    # Convert timestamp to hour:minute AM/PM
    df['timestamp'] = pd.to_datetime(
        df['timestamp'],
        format='%m/%d/%Y %H:%M'
    )

    df['timestamp'] = df['timestamp'].dt.strftime('%I:%M %p')

    # Remove unnecessary columns
    df = df.drop(
        columns=['sensor_id', 'sensor_label'],
        errors='ignore'
    )

    # Rename columns
    df = df.rename(columns={
        'timestamp': 'time',
        'elapsed_time_in_Min': 'elapsed_Mins',
        'Timer_in_Min': 'CP_timer',
        'temperature_c': 'temp_C',
        'temperature_f': 'temp_F'
    })

    # Reset index
    df = df.reset_index(drop=True)

    # Create output filename
    output_path = file_path.with_name(
        f"{file_path.stem}_clean.csv"
    )

    # Export cleaned CSV
    df.to_csv(output_path, index=False)

    print(f"Cleaned CSV saved to: {output_path}")

    return df


import pandas as pd
import numpy as np
import plotly.graph_objects as go


def plot_sauna_temp(
    df,
    temp_col,
    model,
    serial_number,
    test_date,
    interval=5,
    marker_size=14,
    show_time=False,
    dot_color='#F05A40',
    line_color='#4F67FF'
):
    """
    Sauna temperature plot.

    Required:
        elapsed_Mins, CP_timer, temp_F/temp_C

    Optional:
        time (only required when show_time=True)

    Custom colors:
        dot_color  = marker color
        line_color = temperature line color
    """

    # Validate
    required = ['elapsed_Mins', 'CP_timer', temp_col]

    if show_time:
        required.append('time')

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Unit
    unit = '°F' if temp_col == 'temp_F' else '°C' if temp_col == 'temp_C' else '°'

    # Clean
    df = df.copy()
    df['elapsed_Mins'] = pd.to_numeric(df['elapsed_Mins'], errors='coerce')
    df = df.dropna(subset=['elapsed_Mins', temp_col])
    df = df.sort_values('elapsed_Mins').reset_index(drop=True)

    # Uniform interval positions
    max_elapsed = df['elapsed_Mins'].max()
    tick_positions = np.arange(0, max_elapsed + interval, interval)

    # Nearest actual measurement to each interval
    rows = []

    for x in tick_positions:
        idx = (df['elapsed_Mins'] - x).abs().idxmin()
        row = df.loc[idx].copy()
        row['plot_x'] = x
        rows.append(row)

    df_plot = pd.DataFrame(rows)

    # Theme
    navy = '#203A5F'
    plot_bg = '#E7EDF6'
    band = '#F1F5FA'
    grid = '#FFFFFF'
    gray = '#4B5563'

    fig = go.Figure()

    # Temperature line
    fig.add_trace(go.Scatter(
        x=df['elapsed_Mins'],
        y=df[temp_col],
        mode='lines',
        line=dict(width=3, color=line_color, shape='spline'),
        hovertemplate=(
            '<b>Elapsed:</b> %{x:.1f} min<br>'
            f'<b>Temperature:</b> %{{y:.1f}}{unit}'
            '<extra></extra>'
        ),
        showlegend=False
    ))

    # Hover data
    hover_cols = ['time', 'CP_timer', 'elapsed_Mins'] if show_time else ['CP_timer', 'elapsed_Mins']

    if show_time:
        hover_template = (
            '<b>Time:</b> %{customdata[0]}<br>'
            '<b>Elapsed:</b> %{customdata[2]:.1f} min<br>'
            '<b>CP Timer:</b> %{customdata[1]:.0f}<br>'
            f'<b>Temperature:</b> %{{y:.1f}}{unit}'
            '<extra></extra>'
        )
    else:
        hover_template = (
            '<b>Elapsed:</b> %{customdata[1]:.1f} min<br>'
            '<b>CP Timer:</b> %{customdata[0]:.0f}<br>'
            f'<b>Temperature:</b> %{{y:.1f}}{unit}'
            '<extra></extra>'
        )

    # Markers + labels
    fig.add_trace(go.Scatter(
        x=df_plot['plot_x'],
        y=df_plot[temp_col],
        mode='markers+text',
        marker=dict(
            size=marker_size,
            color=dot_color,
            line=dict(width=1.5, color='white')
        ),
        text='<b>' + df_plot[temp_col].round().astype(int).astype(str) + f'{unit}</b>',
        textposition='top center',
        textfont=dict(size=15, color=navy),
        customdata=df_plot[hover_cols],
        hovertemplate=hover_template,
        showlegend=False
    ))

    # X-axis
    fig.update_xaxes(
        tickmode='array',
        tickvals=tick_positions,
        showticklabels=False,
        ticks='',
        showline=True,
        linewidth=1.5,
        linecolor=navy,
        showgrid=True,
        gridcolor=grid,
        gridwidth=1,
        zeroline=False,
        range=[-interval * 0.4, tick_positions[-1] + interval * 0.4]
    )

    # Y-axis
    fig.update_yaxes(
        title=dict(
            text=f'<b>Temperature ({unit})</b>',
            font=dict(size=18, color=navy)
        ),
        tickfont=dict(size=13, color=navy),
        ticks='outside',
        ticklen=5,
        tickcolor=navy,
        showline=True,
        linewidth=1.5,
        linecolor=navy,
        showgrid=True,
        gridcolor=grid,
        gridwidth=1,
        zeroline=False
    )

    # Bottom band
    bottom = -0.20 if show_time else -0.13

    fig.add_shape(
        type='rect',
        x0=0, x1=1,
        y0=bottom, y1=0,
        xref='paper', yref='paper',
        fillcolor=band,
        line=dict(width=0),
        layer='below'
    )

    # Label positions
    elapsed_y = -0.045
    timer_y = -0.095
    time_y = -0.155

    # Bottom values
    for _, row in df_plot.iterrows():

        fig.add_annotation(
            x=row['plot_x'], y=elapsed_y,
            xref='x', yref='paper',
            text=f"<b>{int(row['plot_x'])}</b>",
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(size=13, color=navy)
        )

        fig.add_annotation(
            x=row['plot_x'], y=timer_y,
            xref='x', yref='paper',
            text=f"<b>{row['CP_timer']:.0f}</b>",
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(size=13, color=navy)
        )

        if show_time:
            fig.add_annotation(
                x=row['plot_x'], y=time_y,
                xref='x', yref='paper',
                text=str(row['time']),
                showarrow=False,
                xanchor='center',
                yanchor='middle',
                font=dict(size=12, color=gray)
            )

    # Right-side labels
    labels = [
        (elapsed_y, 'Elapsed'),
        (timer_y, 'CP Timer')
    ]

    if show_time:
        labels.append((time_y, 'Time'))

    for y, label in labels:
        fig.add_annotation(
            x=1.015, y=y,
            xref='paper', yref='paper',
            text=f'<b>{label}</b>',
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            font=dict(size=13, color=navy)
        )

    # Separator
    fig.add_shape(
        type='line',
        x0=0, x1=1,
        y0=-0.07, y1=-0.07,
        xref='paper', yref='paper',
        line=dict(color='#D9E1EC', width=1)
    )

    # Header accent
    fig.add_shape(
        type='line',
        x0=0, x1=1,
        y0=1.04, y1=1.04,
        xref='paper', yref='paper',
        line=dict(color=line_color, width=3)
    )

    # Layout
    fig.update_layout(
        title=dict(
            text=f'<b>{model} - SN {serial_number} | Date {test_date}</b>',
            x=0.5,
            xanchor='center',
            y=0.97,
            font=dict(size=22, color=navy)
        ),
        font=dict(family='Arial', size=14, color=navy),
        height=700,
        paper_bgcolor='white',
        plot_bgcolor=plot_bg,
        margin=dict(l=120, r=150, t=110, b=155 if show_time else 120),
        hovermode='closest',
        hoverlabel=dict(
            bgcolor='white',
            bordercolor='#D5DDEA',
            font=dict(size=13, color=navy)
        ),
        showlegend=False
    )



    return fig

In [5]:
file_path = '/content/2026-09-17_834AM_CS.2_73252-3403_Luis_.csv'

df = process_sauna_data(file_path)

# Reformat the 'time' column to remove AM/PM
df['time'] = pd.to_datetime(df['time'], format='%I:%M %p').dt.strftime('%H:%M')

display(df.head())

Cleaned CSV saved to: /content/2026-09-17_834AM_CS.2_73252-3403_Luis__clean.csv


,time,elapsed_Mins,CP_timer,temp_C,temp_F
0,08:20,0,90,24.2,75.5
1,08:21,1,89,24.4,76.0
2,08:22,2,88,24.5,76.1
3,08:23,3,87,24.6,76.3
4,08:24,4,86,24.6,76.3


### Data Visualization: Temperature Trends

Below are three plots illustrating the `temp_F` (temperature in Fahrenheit) against different time-related metrics: `time`, `elapsed_Mins`, and `CP_timer`. This will help in understanding how the temperature changes over the course of the sauna session.

In [17]:
plot_sauna_temp(
    df,
    temp_col='temp_F',
    model='CS.2 Luis',
    serial_number='73252_3403',
    test_date='09/17/2026',
    dot_color='green',
    line_color='darkgreen'
)

In [16]:
plot_sauna_temp(
    df,
    temp_col='temp_C',
    model='CS.2 Luis',
    serial_number='73252_3403',
    test_date='09/17/2026',
    dot_color='red',
    line_color='darkred'
)